# Clasificador de botellas

Misma red que `Clasificador_de_perros_y_gatos.ipynb`, con otro problema binario:

- cerca de **0** → no botella
- cerca de **1** → botella

Entrada **100×100×1**, tres convoluciones **32 / 64 / 128**, MaxPool, Flatten, Dense 100 ReLU y Dense 1 sigmoide.

Se guarda como `botellas-cnn-ad.h5` y se convierte a `web/modelo_botellas/`.
Esa carpeta no sustituye a `web/modelo/` (perros y gatos).

Este cuaderno está pensado para **Google Colab**. No se ejecuta aquí: la compilación de los pesos es el último bloque.

In [ ]:
!pip install -q tensorflow tensorflowjs opencv-python-headless kagglehub

## Datos

Hace falta una cuenta de Kaggle y el archivo `kaggle.json` (Account → Create New Token).
En Colab se sube a `~/.kaggle/kaggle.json`.

Conjuntos útiles, todos con fotos de botellas de plástico:

| Nombre | Slug en Kaggle | Cómo se parte |
|---|---|---|
| Drinking waste (PET frente al resto) | `arkadiyhacks/drinking-waste-classification` | PET = botella; aluminio y vidrio = no botella |
| Bottle Detection / YOLO | el slug que descargues | recortes con caja = botella; recortes sin caja = no botella |
| Plastic Bottle Images | el slug de las 70k | hace falta una carpeta propia de fondos |

La celda siguiente usa el primero porque ya trae las dos clases. Si cambias de dataset, ajusta `CLASE_BOTELLA`.

In [ ]:
from pathlib import Path
import os
import shutil
import cv2
import numpy as np

DATASET = "arkadiyhacks/drinking-waste-classification"
CLASE_BOTELLA = "PET"
ANCHO = 100
ALTO = 100
RAIZ = Path("/content/botellas")
POSITIVAS = RAIZ / "botella"
NEGATIVAS = RAIZ / "no_botella"

for carpeta in (POSITIVAS, NEGATIVAS):
    if carpeta.exists():
        shutil.rmtree(carpeta)
    carpeta.mkdir(parents=True)

import kagglehub
origen = Path(kagglehub.dataset_download(DATASET))
print("Descargado en", origen)

def a_gris(ruta):
    """Misma llamada que el cuaderno de perros y gatos: COLOR_BGR2GRAY.
    La página aplica esos pesos (0.114, 0.587, 0.299) sobre el RGB del canvas.
    """
    imagen = cv2.imread(str(ruta))
    if imagen is None:
        return None
    gris = cv2.cvtColor(imagen, cv2.COLOR_BGR2GRAY)
    return cv2.resize(gris, (ANCHO, ALTO))

extensiones = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
copiadas = {"botella": 0, "no_botella": 0}
for foto in origen.rglob("*"):
    if not foto.is_file() or foto.suffix.lower() not in extensiones:
        continue
    destino_nombre = "botella" if CLASE_BOTELLA.lower() in str(foto.parent).lower() else "no_botella"
    gris = a_gris(foto)
    if gris is None:
        continue
    salida = (POSITIVAS if destino_nombre == "botella" else NEGATIVAS) / f"{copiadas[destino_nombre]:05d}.png"
    cv2.imwrite(str(salida), gris)
    copiadas[destino_nombre] += 1

print(copiadas)
if copiadas["botella"] < 50 or copiadas["no_botella"] < 50:
    raise SystemExit("Faltan fotos en alguna clase. Revisa CLASE_BOTELLA o el slug del dataset.")

## Aumento de datos

Giro, desplazamiento, recorte (`shear`), zoom y volteo horizontal y vertical.
Las imágenes ya están en gris, así que el generador no vuelve a colorearlas.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

aumento = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode="nearest",
    validation_split=0.15,
)

comun = dict(target_size=(ALTO, ANCHO), color_mode="grayscale", class_mode="binary", batch_size=32, seed=42)
entrenamiento = aumento.flow_from_directory(RAIZ, subset="training", **comun)
validacion = aumento.flow_from_directory(RAIZ, subset="validation", **comun)
print("Índices de clase:", entrenamiento.class_indices)
print("La sigmoide debe acercarse a 1 en la clase botella. Si el índice sale al revés, invierte el umbral en la página.")

## Red `modeloCNN_AD`

Adam, `binary_crossentropy`, exactitud. 150 épocas y lotes de 32, como en el cuaderno de mascotas.

In [ ]:
import tensorflow as tf

modeloCNN_AD = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(100, 100, 1), name="image_input"),
    tf.keras.layers.Conv2D(32, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Conv2D(64, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Conv2D(128, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(100, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
], name="botellas_classifier")

modeloCNN_AD.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
modeloCNN_AD.summary()

historia = modeloCNN_AD.fit(entrenamiento, validation_data=validacion, epochs=150)
modeloCNN_AD.save("botellas-cnn-ad.h5")
print("Guardado: botellas-cnn-ad.h5")

## Compilar a TensorFlow.js

Este es el paso que produce los pesos. Copia después el contenido de `web/modelo_botellas/` al proyecto, **al lado** de `web/modelo/`, sin reemplazarlo.

Cuando existan los `.bin`, añádelos a `APP_FILES` en `web/service-worker.js` para que también queden sin conexión.

In [ ]:
SALIDA = Path("web/modelo_botellas")
if SALIDA.exists():
    shutil.rmtree(SALIDA)
SALIDA.mkdir(parents=True)

!tensorflowjs_converter --input_format keras botellas-cnn-ad.h5 web/modelo_botellas

print(sorted(p.name for p in SALIDA.iterdir()))